# Task 5 – Exploratory Data Analysis (EDA)

**Dataset:** Titanic passenger dataset (`train.csv`)  
**Objective:** Extract insights using visual and statistical exploration.

This notebook follows the Task 5 requirements: `.describe()`, `.info()`, `.value_counts()`, histograms, boxplots, scatterplots, correlation heatmap, pairplot, observations, and a summary of findings.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

df = pd.read_csv("train.csv")
df.head()

## 1. Dataset Overview

In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
# Frequency summaries for important categorical variables
print("Sex:")
display(df["Sex"].value_counts())
print("\nPassenger Class:")
display(df["Pclass"].value_counts().sort_index())
print("\nEmbarked:")
display(df["Embarked"].value_counts())

### Initial observations
- The dataset contains **891 passengers and 12 columns**.
- `Survived` is the main outcome: 0 = did not survive, 1 = survived.
- `Age` has missing values, while `Cabin` has substantial missingness.
- `Embarked` has only a small number of missing values.
- `Fare` has a wide range, indicating possible skew and outliers.

## 2. Missing-Value Analysis

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_table = pd.DataFrame({"Missing Values": missing, "Missing %": missing_pct})
missing_table[missing_table["Missing Values"] > 0]

### Observation
`Cabin` is missing for most passengers, `Age` is missing for a meaningful portion, and `Embarked` has only two missing records. For this EDA, the original dataset is retained unchanged. When numerical relationships require complete rows, a separate analysis copy is created rather than overwriting the source data.

## 3. Univariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="Survived", color="steelblue", ax=axes[0])
axes[0].set_title("Survival Distribution")
axes[0].set_xlabel("Survived (0=No, 1=Yes)")
axes[0].set_ylabel("Passengers")

sns.histplot(data=df, x="Age", bins=30, kde=True, color="steelblue", ax=axes[1])
axes[1].set_title("Age Distribution")
axes[1].set_xlabel("Age")
axes[1].set_ylabel("Frequency")
plt.tight_layout()
plt.show()

### Observations
- More passengers did not survive than survived.
- The overall survival rate is about **38.38%**.
- The age distribution is concentrated around young and middle-aged adults, with fewer very young and elderly passengers.

## 4. Bivariate Analysis

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="Sex", hue="Survived", palette="Set2")
plt.title("Survival by Sex")
plt.xlabel("Sex")
plt.ylabel("Passengers")
plt.show()

sex_survival = pd.crosstab(df["Sex"], df["Survived"], normalize="index").round(3)
sex_survival

### Observation
Survival differs strongly by sex in this dataset. Approximately **74.2% of female passengers** survived compared with **18.9% of male passengers**.

In [ ]:
plt.figure(figsize=(7, 5))
sns.barplot(data=df, x="Pclass", y="Survived", errorbar=None, color="steelblue")
plt.title("Survival Rate by Passenger Class")
plt.xlabel("Passenger Class")
plt.ylabel("Survival Rate")
plt.ylim(0, 0.75)
plt.show()

class_survival = df.groupby("Pclass")["Survived"].mean().round(3)
class_survival

### Observation
Passenger class is strongly associated with survival. First-class passengers had the highest survival rate (**63.0%**), followed by second class (**47.3%**) and third class (**24.2%**).

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Pclass", y="Fare", color="lightgray")
plt.title("Fare Distribution by Passenger Class")
plt.xlabel("Passenger Class")
plt.ylabel("Fare")
plt.show()

### Observation
Fares are much higher and more variable in first class. The boxplot also shows high-fare observations that can influence averages and correlations.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="Age", y="Fare", hue="Survived", alpha=0.65, palette="Set1")
plt.title("Age vs Fare by Survival")
plt.xlabel("Age")
plt.ylabel("Fare")
plt.show()

### Observation
There is no strong simple linear relationship between age and fare. Higher fares are concentrated among certain passengers, particularly those associated with first class, and survival groups overlap substantially.

## 5. Correlation Analysis

In [ ]:
numeric_cols = ["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare"]
corr = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()

corr["Survived"].sort_values(ascending=False)

### Observation
- `Pclass` has a negative correlation with survival (**-0.34**): higher class number means lower passenger class and is associated with lower survival.
- `Fare` has a positive correlation with survival (**0.26**).
- `Age`, `SibSp`, and `Parch` show relatively weak linear correlations with survival.
- Correlation measures linear association and does not prove causation.

## 6. Pairplot

In [ ]:
pair_df = df[numeric_cols].dropna().sample(n=min(300, len(df[numeric_cols].dropna())), random_state=42)
sns.pairplot(pair_df, hue="Survived", corner=True, palette="Set1")
plt.suptitle("Pairplot of Key Numeric Variables", y=1.02)
plt.show()

### Observation
The pairplot gives a multivariate view of the numeric variables. The clearest separation of survival groups is associated with `Pclass` and, to a lesser extent, `Fare`; the remaining variables show considerable overlap.

## 7. Skewness and Outlier Check

In [ ]:
skewness = df.select_dtypes(include=np.number).skew().sort_values(ascending=False)
skewness

### Observation and handling approach
`Fare` is highly right-skewed, while `SibSp` and `Parch` are also right-skewed. For a predictive modeling workflow, skewed positive variables can be transformed using a method such as `log1p` (when appropriate). For this EDA, the raw values are retained so that the original distribution and outliers remain visible.

In [ ]:
# Example transformation for analysis only; the original Fare column is unchanged.
eda_copy = df.copy()
eda_copy["Fare_log1p"] = np.log1p(eda_copy["Fare"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(eda_copy["Fare"], bins=30, kde=True, color="steelblue", ax=axes[0])
axes[0].set_title("Original Fare")
sns.histplot(eda_copy["Fare_log1p"], bins=30, kde=True, color="steelblue", ax=axes[1])
axes[1].set_title("log1p(Fare)")
plt.tight_layout()
plt.show()

## 8. Multicollinearity

Multicollinearity means strong correlation among predictor variables. A correlation heatmap is a useful first check; for a formal modeling workflow, **Variance Inflation Factor (VIF)** can also be calculated after preparing numeric predictors. In this dataset, `Pclass` and `Fare` have a noticeable inverse/positive relationship with each other, reflecting passenger class and fare structure, but the heatmap does not show extreme pairwise correlations among the selected numeric predictors.

In [ ]:
predictors = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
corr[predictors].loc[predictors]

## 9. Key Findings

1. The dataset has **891 passengers, 12 variables**, and a survival rate of **38.38%**.
2. **Sex** is strongly associated with survival: female survival is much higher than male survival.
3. **Passenger class** is an important pattern: first class has the highest survival rate and third class the lowest.
4. **Fare** is strongly right-skewed and contains high-value outliers.
5. `Pclass` has the strongest negative linear correlation with survival among the selected numeric variables, while `Fare` has the strongest positive correlation.
6. `Age`, `SibSp`, and `Parch` have comparatively weak linear correlations with survival.
7. Missing values are concentrated in `Cabin` and `Age`; `Embarked` has only two missing values.
8. EDA reveals associations and patterns, but these observations alone do not establish causation.

## 10. Interview Questions – Short Answers

**1. What is EDA and why is it important?**  
EDA is the process of exploring data using statistics and visualizations to understand distributions, missing values, outliers, relationships, and patterns before modeling.

**2. Which plots do you use to check correlation?**  
A correlation heatmap is useful for many numeric variables. Scatterplots are useful for examining the relationship between two numeric variables.

**3. How do you handle skewed data?**  
First identify skewness using statistics and plots. Depending on the use case, transformations such as `log1p`, square root, or other suitable transformations can reduce skewness. The choice should preserve interpretability and fit the analysis goal.

**4. How to detect multicollinearity?**  
Use a correlation matrix/heatmap for an initial check and VIF for a formal diagnostic among predictors.

**5. What are univariate, bivariate, and multivariate analyses?**  
Univariate examines one variable, bivariate examines the relationship between two variables, and multivariate examines multiple variables together.

**6. Difference between heatmap and pairplot?**  
A heatmap summarizes relationships numerically using color-coded matrix cells. A pairplot shows pairwise distributions and scatterplots for several variables.

**7. How do you summarize your insights?**  
State the most important patterns, compare meaningful groups, mention missing values/outliers, and explain what the visuals and statistics indicate without claiming unsupported causation.

## Conclusion

The Titanic dataset demonstrates how EDA can uncover meaningful patterns in passenger survival. The strongest visible differences are associated with **sex and passenger class**, while fare shows skewness and outliers that should be considered during further analysis. This notebook provides a reproducible EDA workflow using Pandas, Matplotlib, and Seaborn.